# 遷移學習與微調

遷移學習(Transfer Learning)是深度學習中最實用的技術之一。它允許我們利用在大型數據集（如ImageNet）上預訓練的模型來解決自己的問題。

## 為什麼需要遷移學習？

1. **數據不足**：從零訓練需要大量數據，而遷移學習可以在小數據集上取得好效果
2. **計算資源有限**：預訓練模型已經學到了通用特徵，無需從頭訓練
3. **訓練時間**：大幅減少訓練時間
4. **更好的性能**：預訓練模型的特徵提取能力通常優於隨機初始化

## 本章內容

1. **遷移學習基礎** - 概念和原理
2. **特徵提取** - 固定預訓練模型，只訓練分類器
3. **微調** - 解凍部分或全部層進行訓練
4. **實戰案例** - 在自定義數據集上應用遷移學習
5. **最佳實踐** - 超參數選擇、學習率策略

## 學習目標

- 理解遷移學習的原理和適用場景
- 掌握PyTorch中的遷移學習實現
- 學會選擇和調整預訓練模型
- 理解何時使用特徵提取vs微調

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import transforms, models, datasets
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import time
import copy
from tqdm import tqdm

# 設置隨機種子
torch.manual_seed(42)
np.random.seed(42)

# 設備配置
device = torch.device('cuda' if torch.cuda.is_available() else 
                     'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'使用設備: {device}')

## 1. 遷移學習基礎

### 核心思想

在大型數據集（如ImageNet）上訓練的CNN學到了通用的視覺特徵：
- **淺層**：邊緣、顏色、紋理等低級特徵
- **深層**：形狀、物體部件等高級特徵

這些特徵對於其他視覺任務也是有用的！

### 兩種策略

#### 1. 特徵提取 (Feature Extraction)
```
預訓練模型（凍結）→ 特徵提取 → 新的分類器（訓練）
```
- 固定預訓練模型的所有層
- 只訓練新添加的分類器層
- 適用於：數據集小、與預訓練數據相似

#### 2. 微調 (Fine-tuning)
```
預訓練模型（部分解凍）→ 繼續訓練 → 適應新任務
```
- 解凍部分或全部層
- 使用較小的學習率繼續訓練
- 適用於：數據集較大、與預訓練數據有一定差異

### 選擇指南

| 數據集大小 | 數據相似度 | 推薦策略 |
|------------|------------|----------|
| 小 | 高 | 特徵提取 |
| 小 | 低 | 特徵提取 + 微調少數層 |
| 大 | 高 | 微調 |
| 大 | 低 | 完全微調 或 從頭訓練 |

## 2. 載入預訓練模型

PyTorch提供了多種預訓練模型：
- **經典模型**：AlexNet, VGG, ResNet, DenseNet
- **輕量級模型**：MobileNet, ShuffleNet, SqueezeNet
- **現代模型**：EfficientNet, RegNet, Vision Transformer

In [ ]:
# 載入預訓練的ResNet-18
print("載入預訓練的ResNet-18模型...")
resnet18 = models.resnet18(pretrained=True)

print("\nResNet-18架構:")
print(resnet18)

# 查看模型的輸入要求
print("\n注意：ImageNet預訓練模型通常要求：")
print("- 輸入尺寸: 224×224")
print("- 輸入通道: 3 (RGB)")
print("- 輸出類別: 1000 (ImageNet類別數)")
print(f"\n當前模型最後一層: {resnet18.fc}")

## 3. 特徵提取

### 步驟

1. 載入預訓練模型
2. 凍結所有層的參數
3. 替換最後的分類器層
4. 只訓練新的分類器層

In [ ]:
def set_parameter_requires_grad(model, feature_extracting):
    """
    凍結或解凍模型參數
    
    Args:
        model: 模型
        feature_extracting: True表示凍結參數
    """
    if feature_extracting:
        for param in model.parameters():
            param.requires_grad = False

def create_feature_extractor(model_name='resnet18', num_classes=10, use_pretrained=True):
    """
    創建特徵提取器
    
    Args:
        model_name: 模型名稱
        num_classes: 目標類別數
        use_pretrained: 是否使用預訓練權重
    """
    model_ft = None
    input_size = 224
    
    if model_name == "resnet18":
        # 載入預訓練的ResNet-18
        model_ft = models.resnet18(pretrained=use_pretrained)
        
        # 凍結所有層
        set_parameter_requires_grad(model_ft, feature_extracting=True)
        
        # 替換最後的全連接層
        num_ftrs = model_ft.fc.in_features
        model_ft.fc = nn.Linear(num_ftrs, num_classes)
    
    elif model_name == "vgg16":
        model_ft = models.vgg16(pretrained=use_pretrained)
        set_parameter_requires_grad(model_ft, feature_extracting=True)
        
        # VGG的分類器是一個Sequential
        num_ftrs = model_ft.classifier[6].in_features
        model_ft.classifier[6] = nn.Linear(num_ftrs, num_classes)
    
    elif model_name == "mobilenet_v2":
        model_ft = models.mobilenet_v2(pretrained=use_pretrained)
        set_parameter_requires_grad(model_ft, feature_extracting=True)
        
        num_ftrs = model_ft.classifier[1].in_features
        model_ft.classifier[1] = nn.Linear(num_ftrs, num_classes)
    
    return model_ft, input_size

# 創建特徵提取器
model_ft, input_size = create_feature_extractor('resnet18', num_classes=10)
model_ft = model_ft.to(device)

# 檢查哪些參數需要訓練
params_to_update = []
print("需要訓練的參數:")
for name, param in model_ft.named_parameters():
    if param.requires_grad:
        params_to_update.append(param)
        print(f"  {name}")

print(f"\n總共 {len(params_to_update)} 個參數需要訓練")
print(f"新的分類器: {model_ft.fc}")

## 4. 微調 (Fine-tuning)

### 策略

1. **微調最後幾層**：最常用，保持底層特徵不變
2. **微調所有層**：數據集較大時
3. **差異化學習率**：底層用小學習率，頂層用大學習率

In [ ]:
def create_finetune_model(model_name='resnet18', num_classes=10, 
                         freeze_layers=True, freeze_until_layer=6):
    """
    創建微調模型
    
    Args:
        model_name: 模型名稱
        num_classes: 目標類別數
        freeze_layers: 是否凍結部分層
        freeze_until_layer: 凍結到第幾層（ResNet的layer編號）
    """
    model_ft = models.resnet18(pretrained=True)
    
    if freeze_layers:
        # 凍結早期層
        layer_count = 0
        for name, child in model_ft.named_children():
            layer_count += 1
            if layer_count <= freeze_until_layer:
                print(f"凍結層: {name}")
                for param in child.parameters():
                    param.requires_grad = False
            else:
                print(f"解凍層: {name}")
    
    # 替換最後的全連接層
    num_ftrs = model_ft.fc.in_features
    model_ft.fc = nn.Linear(num_ftrs, num_classes)
    
    return model_ft

# 創建微調模型
print("=== 創建微調模型 ===")
model_finetune = create_finetune_model('resnet18', num_classes=10, 
                                      freeze_layers=True, freeze_until_layer=6)
model_finetune = model_finetune.to(device)

# 統計可訓練參數
trainable_params = sum(p.numel() for p in model_finetune.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model_finetune.parameters())
print(f"\n可訓練參數: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.1f}%)")

## 5. 差異化學習率

為不同層設置不同的學習率：
- **底層（預訓練特徵）**：小學習率（如1e-5）
- **頂層（新分類器）**：大學習率（如1e-3）

In [ ]:
def get_optimizer_with_differential_lr(model, base_lr=1e-4, fc_lr=1e-3):
    """
    創建具有差異化學習率的優化器
    
    Args:
        model: 模型
        base_lr: 預訓練層的學習率
        fc_lr: 新分類器層的學習率
    """
    # 分離參數組
    pretrained_params = []
    new_params = []
    
    for name, param in model.named_parameters():
        if param.requires_grad:
            if 'fc' in name:  # ResNet的全連接層名為'fc'
                new_params.append(param)
            else:
                pretrained_params.append(param)
    
    # 創建參數組
    param_groups = [
        {'params': pretrained_params, 'lr': base_lr},
        {'params': new_params, 'lr': fc_lr}
    ]
    
    optimizer = optim.Adam(param_groups)
    
    print(f"優化器配置:")
    print(f"  預訓練層: {len(pretrained_params)} 個參數組, 學習率 = {base_lr}")
    print(f"  新分類器: {len(new_params)} 個參數組, 學習率 = {fc_lr}")
    
    return optimizer

# 創建差異化學習率的優化器
optimizer_diff = get_optimizer_with_differential_lr(model_finetune, base_lr=1e-4, fc_lr=1e-3)

## 6. 完整的訓練流程

### 數據預處理

使用ImageNet的歸一化參數：
- mean = [0.485, 0.456, 0.406]
- std = [0.229, 0.224, 0.225]

In [ ]:
# ImageNet的預處理參數
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

# 訓練時的數據增強
train_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

# 驗證/測試時的預處理
val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

print("數據變換已設置")
print("\n訓練變換:")
print(train_transforms)
print("\n驗證變換:")
print(val_transforms)

In [ ]:
def train_model(model, dataloaders, criterion, optimizer, num_epochs=10, 
               scheduler=None, is_inception=False):
    """
    訓練模型
    
    Args:
        model: 要訓練的模型
        dataloaders: 包含'train'和'val'的數據加載器字典
        criterion: 損失函數
        optimizer: 優化器
        num_epochs: 訓練輪數
        scheduler: 學習率調度器
        is_inception: 是否是Inception模型（有輔助輸出）
    """
    since = time.time()
    
    # 記錄歷史
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }
    
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    
    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 60)
        
        # 每個epoch都有訓練和驗證階段
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  # 設置為訓練模式
            else:
                model.eval()   # 設置為評估模式
            
            running_loss = 0.0
            running_corrects = 0
            
            # 遍歷數據
            for inputs, labels in tqdm(dataloaders[phase], desc=phase):
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                # 清零梯度
                optimizer.zero_grad()
                
                # 前向傳播
                with torch.set_grad_enabled(phase == 'train'):
                    # Inception模型在訓練時有輔助輸出
                    if is_inception and phase == 'train':
                        outputs, aux_outputs = model(inputs)
                        loss1 = criterion(outputs, labels)
                        loss2 = criterion(aux_outputs, labels)
                        loss = loss1 + 0.4 * loss2
                    else:
                        outputs = model(inputs)
                        loss = criterion(outputs, labels)
                    
                    _, preds = torch.max(outputs, 1)
                    
                    # 反向傳播
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()
                
                # 統計
                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
            
            if phase == 'train' and scheduler is not None:
                scheduler.step()
            
            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects.double() / len(dataloaders[phase].dataset)
            
            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            
            # 記錄歷史
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(epoch_acc.item())
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(epoch_acc.item())
            
            # 深拷貝最佳模型
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
        
        print()
    
    time_elapsed = time.time() - since
    print(f'訓練完成，耗時 {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'最佳驗證準確率: {best_acc:.4f}')
    
    # 載入最佳模型權重
    model.load_state_dict(best_model_wts)
    return model, history

print("訓練函數已定義")

## 7. 實戰示例：在CIFAR-10上使用遷移學習

由於CIFAR-10圖像尺寸為32×32，需要調整預處理：
1. 上採樣到224×224（或其他適合的尺寸）
2. 或修改模型的第一層以接受32×32輸入

In [ ]:
# CIFAR-10的數據變換
cifar_train_transforms = transforms.Compose([
    transforms.Resize(224),  # 上採樣到224×224
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(224, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

cifar_val_transforms = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

# 載入CIFAR-10數據集
print("載入CIFAR-10數據集...")
train_dataset = datasets.CIFAR10(root='./data', train=True, 
                                download=True, transform=cifar_train_transforms)
val_dataset = datasets.CIFAR10(root='./data', train=False,
                              download=True, transform=cifar_val_transforms)

# 創建數據加載器
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, 
                         shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, 
                       shuffle=False, num_workers=2)

dataloaders = {'train': train_loader, 'val': val_loader}

print(f"訓練集大小: {len(train_dataset)}")
print(f"驗證集大小: {len(val_dataset)}")

# CIFAR-10類別名稱
classes = ('plane', 'car', 'bird', 'cat', 'deer', 
          'dog', 'frog', 'horse', 'ship', 'truck')
print(f"類別: {classes}")

## 8. 遷移學習最佳實踐

### 1. 選擇合適的預訓練模型

- **準確度優先**：ResNet-50/101, EfficientNet-B3/B4
- **速度優先**：MobileNetV2, EfficientNet-B0
- **記憶體受限**：MobileNetV2, SqueezeNet

### 2. 學習率策略

- **特徵提取**：較大學習率（1e-3）
- **微調**：較小學習率（1e-4 ~ 1e-5）
- **差異化學習率**：底層1e-5，頂層1e-3

### 3. 訓練策略

**兩階段訓練**（推薦）：
1. 階段1：凍結預訓練層，訓練分類器（5-10 epochs）
2. 階段2：解凍部分層，使用小學習率微調（10-20 epochs）

### 4. 數據增強

- 數據集小：強數據增強
- 數據集大：輕度數據增強
- 必須使用ImageNet的歸一化參數

### 5. 常見問題

**Q: 驗證準確率很低？**
- 檢查是否使用了正確的歸一化參數
- 檢查數據預處理是否正確
- 嘗試降低學習率

**Q: 過擬合？**
- 增加數據增強
- 使用Dropout
- 減少微調的層數
- 使用Weight Decay

**Q: 訓練太慢？**
- 使用更小的模型
- 減少輸入尺寸
- 使用混合精度訓練
- 增加batch size

## 9. 實用技巧

### 逐步解凍 (Gradual Unfreezing)

In [ ]:
def gradual_unfreeze(model, unfreeze_layer_names):
    """
    逐步解凍指定的層
    
    Args:
        model: 模型
        unfreeze_layer_names: 要解凍的層名稱列表
    """
    for name, param in model.named_parameters():
        for layer_name in unfreeze_layer_names:
            if layer_name in name:
                param.requires_grad = True
                print(f"解凍: {name}")
                break

# 示例：逐步解凍策略
def train_with_gradual_unfreezing(model, dataloaders, num_classes=10):
    """
    使用逐步解凍策略訓練
    """
    criterion = nn.CrossEntropyLoss()
    
    # 階段1：只訓練分類器（5 epochs）
    print("\n=== 階段1：訓練分類器 ===")
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
    model, history1 = train_model(model, dataloaders, criterion, optimizer, num_epochs=5)
    
    # 階段2：解凍最後一個block，微調（5 epochs）
    print("\n=== 階段2：微調最後一個block ===")
    gradual_unfreeze(model, ['layer4'])  # ResNet的最後一個block
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
    model, history2 = train_model(model, dataloaders, criterion, optimizer, num_epochs=5)
    
    # 階段3：解凍更多層，繼續微調（5 epochs）
    print("\n=== 階段3：微調更多層 ===")
    gradual_unfreeze(model, ['layer3'])
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-5)
    model, history3 = train_model(model, dataloaders, criterion, optimizer, num_epochs=5)
    
    # 合併歷史記錄
    history = {
        'train_loss': history1['train_loss'] + history2['train_loss'] + history3['train_loss'],
        'train_acc': history1['train_acc'] + history2['train_acc'] + history3['train_acc'],
        'val_loss': history1['val_loss'] + history2['val_loss'] + history3['val_loss'],
        'val_acc': history1['val_acc'] + history2['val_acc'] + history3['val_acc']
    }
    
    return model, history

print("逐步解凍訓練函數已定義")
print("\n注意：由於訓練時間較長，這裡只提供函數定義。")
print("實際使用時，取消註釋下面的代碼運行訓練。")

# # 運行訓練（取消註釋以執行）
# model_transfer = create_feature_extractor('resnet18', num_classes=10)
# model_transfer = model_transfer.to(device)
# model_trained, history = train_with_gradual_unfreezing(model_transfer, dataloaders)

## 練習題

### 1. 模型對比實驗
比較以下三種策略在CIFAR-10上的性能：
- 從頭訓練
- 特徵提取
- 微調

### 2. 不同模型比較
比較ResNet-18, MobileNetV2, EfficientNet-B0在相同數據集上的遷移學習效果。

### 3. 學習率調優
實驗不同的學習率策略，找出最優配置。

### 4. 數據量影響
使用不同比例的訓練數據（10%, 25%, 50%, 100%），觀察遷移學習的效果。

### 5. 自定義數據集
在自己的數據集上實現遷移學習，並記錄整個過程。

### 6. 多任務學習
使用一個預訓練模型同時預測多個任務（如CIFAR-10分類 + 圖像重建）。

## 總結

### 關鍵要點

1. **遷移學習是實踐中最常用的技術**，可以大幅減少訓練時間和數據需求
2. **選擇合適的策略**：根據數據集大小和相似度選擇特徵提取或微調
3. **學習率很關鍵**：預訓練層用小學習率，新層用大學習率
4. **數據預處理**：必須使用預訓練模型的歸一化參數
5. **逐步解凍**：推薦使用兩階段或多階段訓練策略

### 決策流程

```
開始
  ↓
數據集大嗎？
  ├─ 否 → 與ImageNet相似嗎？
  │        ├─ 是 → 特徵提取
  │        └─ 否 → 特徵提取 + 微調少數層
  │
  └─ 是 → 與ImageNet相似嗎？
           ├─ 是 → 微調（較大學習率）
           └─ 否 → 完全微調（較小學習率）或從頭訓練
```

### 推薦資源

- **預訓練模型**：PyTorch Model Zoo, timm庫
- **數據集**：ImageNet, COCO, Places365
- **工具**：torchvision.models, timm, transformers

### 下一步學習

1. 域適應(Domain Adaptation)
2. 知識蒸餾(Knowledge Distillation)
3. 多任務學習(Multi-task Learning)
4. 元學習(Meta-learning)
5. 自監督學習(Self-supervised Learning)